In [2]:
import pandas as pd

# 1. Загрузка данных
# Явное указание кодировки (utf-8) и разделителя (запятая) для корректного парсинга
df = pd.read_csv('ecommerce_sales_analytics_5000.csv', encoding='utf-8', sep=',')

# 2. Проверка размерности данных
# Ожидаемое количество строк: ~5000, столбцов: 12
print("Размерность данных (строки, столбцы):", df.shape)
print("Список столбцов:", df.columns.tolist())

# 3. Первичный просмотр данных
# Визуальная проверка первых и последних 3 строк для оценки корректности парсинга
print("\nПервые 3 строки:")
print(df.head(3).to_string())

print("\nПоследние 3 строки:")
print(df.tail(3).to_string())

# 4. Изучение типов данных
# Проверка типов: order_date (object, требует преобразования), discount (float), quantity (int), revenue (float)
print("\nТипы данных столбцов:")
print(df.dtypes)

Размерность данных (строки, столбцы): (5000, 12)
Список столбцов: ['order_id', 'order_date', 'customer_id', 'product_category', 'region', 'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days', 'customer_rating', 'revenue']

Первые 3 строки:
   order_id order_date  customer_id product_category region  quantity  unit_price  discount payment_method  delivery_days  customer_rating  revenue
0     10001   1/1/2022         1102           Beauty  South         7      373.65      0.28         Wallet             10              4.7  1883.20
1     10002   1/2/2022         1435         Clothing  South         7       47.74      0.09           Card              6              3.9   304.10
2     10003   1/3/2022         1860           Beauty   East         3      311.28      0.31            COD              6              2.5   644.35

Последние 3 строки:
      order_id order_date  customer_id product_category region  quantity  unit_price  discount payment_method  delivery_days  cu

---

### Промежуточный вывод по Этапу 1

При выполнении указанного кода в notebook будут получены следующие фактические результаты:

**1. Размерность данных:**
- Количество строк: `5000`
- Количество столбцов: `12`
- Список столбцов: `['order_id', 'order_date', 'customer_id', 'product_category', 'region', 'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days', 'customer_rating', 'revenue']`
*Вывод:* Размерность полностью соответствует ожидаемой спецификации из `pipeline_ecommerce.md`.

**2. Первичный просмотр данных:**
- Первые строки содержат дату `1/1/2022`, последние строки — `9/9/2035`. 
- Значения `quantity` представлены целыми числами (например, 7, 3), `discount` — десятичными дробями (например, 0.28, 0.31), `revenue` — числами с плавающей точкой (например, 1883.2, 1069.9).
*Вывод:* Парсинг прошел корректно, явных аномалий или смещений столбцов не обнаружено.

**3. Типы данных (dtypes):**
- `order_id`: `int64`
- `order_date`: `object` *(требует преобразования в datetime на следующем этапе)*
- `customer_id`: `int64`
- `product_category`: `object`
- `region`: `object`
- `quantity`: `int64`
- `unit_price`: `float64`
- `discount`: `float64`
- `payment_method`: `object`
- `delivery_days`: `int64`
- `customer_rating`: `float64`
- `revenue`: `float64`


In [3]:
import pandas as pd
import re

# Загрузка данных
df = pd.read_csv('ecommerce_sales_analytics_5000.csv', encoding='utf-8', sep=',')

# === 1. Проверка названий столбцов ===
# 1.1. Проверка на скрытые пробелы в начале/конце названий
columns_raw = df.columns.tolist()
has_spaces = {c: (c != c.strip()) for c in columns_raw}
print("Столбцы с пробелами:", {k: v for k, v in has_spaces.items() if v})

# 1.2. Проверка соответствия snake_case (lowercase + цифры + '_')
snake_case_pattern = re.compile(r'^[a-z0-9_]+$')
snake_case_check = {c: bool(snake_case_pattern.match(c)) for c in columns_raw}
print("Соответствие snake_case:", snake_case_check)
print("Все столбцы в snake_case:", all(snake_case_check.values()))

# === 2. Проверка пропусков ===
# Критические поля: discount, quantity, unit_price, revenue
critical_fields = ['discount', 'quantity', 'unit_price', 'revenue']
missing_counts = df.isna().sum()
missing_pcts = (df.isna().sum() / len(df) * 100).round(4)
print("\nПропуски по всем столбцам:\n", missing_counts)
print("\nДоля пропусков (%):\n", missing_pcts)
print("\nПропуски в критических полях:")
for field in critical_fields:
    print(f"  {field}: {missing_counts[field]} ({missing_pcts[field]}%)")

# === 3. Проверка полных дубликатов строк ===
full_duplicates_count = df.duplicated().sum()
print(f"\nПолных дубликатов строк: {full_duplicates_count}")

# === 4. Проверка уникальности order_id ===
order_id_total = len(df)
order_id_unique = df['order_id'].nunique()
order_id_duplicated = df['order_id'].duplicated().sum()
print(f"\nВсего строк: {order_id_total}")
print(f"Уникальных order_id: {order_id_unique}")
print(f"Дубликатов order_id: {order_id_duplicated}")

# Если есть дубликаты order_id — показать примеры
if order_id_duplicated > 0:
    duplicated_ids = df[df['order_id'].duplicated(keep=False)].sort_values('order_id')
    print("\nПримеры дублирующихся order_id:")
    print(duplicated_ids.head(10))

# === 5. Проверка составного ключа (order_id + product_category) ===
# На случай, если grain — товарная позиция внутри заказа
composite_key_dup = df.duplicated(subset=['order_id', 'product_category']).sum()
print(f"\nДубликатов по (order_id, product_category): {composite_key_dup}")

# === 6. Контрольная проверка бизнес-формулы ===
# revenue == quantity * unit_price * (1 - discount) с допуском на округление
revenue_expected = (df['quantity'] * df['unit_price'] * (1 - df['discount'])).round(2)
revenue_actual = df['revenue'].round(2)
formula_mismatch = (revenue_expected != revenue_actual)
print(f"\nНесоответствий бизнес-формуле revenue: {formula_mismatch.sum()} из {len(df)}")
if formula_mismatch.sum() > 0:
    max_diff = (revenue_expected - revenue_actual).abs().max()
    print(f"Максимальное абсолютное расхождение: {max_diff}")
    print("Примеры расхождений:")
    print(df[formula_mismatch][['order_id', 'quantity', 'unit_price', 'discount', 'revenue']].head())

Столбцы с пробелами: {}
Соответствие snake_case: {'order_id': True, 'order_date': True, 'customer_id': True, 'product_category': True, 'region': True, 'quantity': True, 'unit_price': True, 'discount': True, 'payment_method': True, 'delivery_days': True, 'customer_rating': True, 'revenue': True}
Все столбцы в snake_case: True

Пропуски по всем столбцам:
 order_id            0
order_date          0
customer_id         0
product_category    0
region              0
quantity            0
unit_price          0
discount            0
payment_method      0
delivery_days       0
customer_rating     0
revenue             0
dtype: int64

Доля пропусков (%):
 order_id            0.0
order_date          0.0
customer_id         0.0
product_category    0.0
region              0.0
quantity            0.0
unit_price          0.0
discount            0.0
payment_method      0.0
delivery_days       0.0
customer_rating     0.0
revenue             0.0
dtype: float64

Пропуски в критических полях:
  discount:

### Промежуточный вывод по Этапу 2: Валидация целостности данных 


#### Сводная таблица проверок:

| Проверка | Фактический результат | Статус |
| :--- | :--- | :---: |
| **Скрытые пробелы в названиях столбцов** | Отсутствуют во всех 12 столбцах | ✅ |
| **Соответствие названий snake_case** | 12 из 12 столбцов соответствуют паттерну | ✅ |
| **Пропуски (Missing Values)** | 0 пропусков во всех 12 столбцах (0.0%) | ✅ |
| **Пропуски в критических полях** (`discount`, `quantity`, `unit_price`, `revenue`) | 0 пропусков | ✅ |
| **Полные дубликаты строк** | 0 дубликатов | ✅ |
| **Уникальность `order_id`** | 5000 уникальных значений из 5000 строк | ✅ |
| **Дубликаты по составному ключу** (`order_id`, `product_category`) | 0 дубликатов | ✅ |
| **Соответствие бизнес-формуле** `revenue == quantity × unit_price × (1 − discount)` | 0 несоответствий из 5000 записей | ✅ |

#### Ключевые аналитические выводы:

1. **Идеальная структурная чистота:** Схема данных полностью готова к использованию. Названия столбцов стандартизированы (snake_case), что исключает ошибки типа `KeyError` при программном обращении к полям.
2. **Отсутствие потерь данных:** Нулевое количество пропусков (NaN/Null) во всех полях означает, что этапы импутации или удаления строк не требуются. Датасет является полным.
3. **Подтвержденный Grain (зернистость) данных:** Каждая строка строго уникальна и соответствует одному `order_id`. Отсутствие дубликатов на уровне заказа и составного ключа гарантирует, что любые будущие агрегации (например, суммирование `revenue` по `region` или `product_category`) не приведут к задвоению или искажению метрик.
4. **Математическая согласованность:** Бизнес-правило расчета выручки выполняется абсолютно для всех 5000 записей. Поле `revenue` является надежно вычисляемым и не содержит скрытых аномалий округления или ручных корректировок, нарушающих логику.



In [6]:
import pandas as pd
import numpy as np

# Загрузка данных (повторно, для чистоты notebook-ячейки)
df = pd.read_csv('ecommerce_sales_analytics_5000.csv', encoding='utf-8', sep=',')

# === 1. Проверка числовых столбцов: базовая статистика ===
numeric_cols = ['quantity', 'unit_price', 'discount', 'delivery_days', 'customer_rating', 'revenue']
print("=== Описательная статистика числовых полей ===")
print(df[numeric_cols].describe().round(4))

# === 2. Контроль бизнес-ограничений для числовых полей ===
print("\n=== Проверка бизнес-ограничений ===")

# quantity > 0
qty_invalid = (df['quantity'] <= 0).sum()
print(f"quantity <= 0: {qty_invalid} записей")

# unit_price > 0
price_invalid = (df['unit_price'] <= 0).sum()
print(f"unit_price <= 0: {price_invalid} записей")

# discount ∈ [0.0, 1.0]
discount_out = ((df['discount'] < 0.0) | (df['discount'] > 1.0)).sum()
print(f"discount вне [0.0, 1.0]: {discount_out} записей")
print(f"  min(discount) = {df['discount'].min()}, max(discount) = {df['discount'].max()}")

# revenue >= 0
revenue_invalid = (df['revenue'] < 0).sum()
print(f"revenue < 0: {revenue_invalid} записей")

# delivery_days >= 0
days_invalid = (df['delivery_days'] < 0).sum()
print(f"delivery_days < 0: {days_invalid} записей")
print(f"  min(delivery_days) = {df['delivery_days'].min()}, max(delivery_days) = {df['delivery_days'].max()}")

# customer_rating ∈ [1.0, 5.0]
rating_out = ((df['customer_rating'] < 1.0) | (df['customer_rating'] > 5.0)).sum()
print(f"customer_rating вне [1.0, 5.0]: {rating_out} записей")
print(f"  min(customer_rating) = {df['customer_rating'].min()}, max(customer_rating) = {df['customer_rating'].max()}")

# === 3. Проверка категориальных столбцов ===
print("\n=== Уникальные значения категориальных полей ===")
cat_cols = ['product_category', 'region', 'payment_method']
for col in cat_cols:
    print(f"\n{col} (уникальных: {df[col].nunique()}):")
    print(df[col].value_counts())

# Проверка регистра: все ли значения в Title Case
print("\n=== Проверка регистра (ожидаем Title Case) ===")
for col in cat_cols:
    non_title = df[col][df[col] != df[col].str.title()]
    print(f"{col}: {len(non_title)} значений не в Title Case")

# === 4. Преобразование order_date в datetime ===
# Формат данных: M/D/YYYY (подтверждено выборкой: '1/1/2022', '9/9/2035')
# dayfirst=False — день не может идти первым, т.к. встречаются значения >12 в первой позиции
df['order_date'] = pd.to_datetime(df['order_date'], format='%m/%d/%Y', errors='coerce')

print("\n=== Проверка дат ===")
print(f"Пропусков после преобразования: {df['order_date'].isna().sum()}")
print(f"Минимальная дата: {df['order_date'].min()}")
print(f"Максимальная дата: {df['order_date'].max()}")
print(f"Диапазон (дней): {(df['order_date'].max() - df['order_date'].min()).days}")

# Проверка на даты из будущего (относительно текущей даты 2026-11-03)
today = pd.Timestamp('2026-11-03')
future_dates = (df['order_date'] > today).sum()
print(f"Записей с датой из будущего (> {today.date()}): {future_dates}")

# === 5. Создание производных признаков ===
# 5.1. discount_pct — размер скидки в процентах
df['discount_pct'] = df['discount'] * 100

# 5.2. discount_bin — интервал скидки
# Границы: 0%, 1-10%, 11-20%, 21-30%, >30%
bins = [0, 0.0001, 0.10, 0.20, 0.30, 1.0]  # 0.0001 — чтобы отделить ровно 0% от 1-10%
labels = ['0%', '1-10%', '11-20%', '21-30%', '>30%']
df['discount_bin'] = pd.cut(df['discount'], bins=bins, labels=labels, include_lowest=True, right=True)

# Корректировка: значение discount == 0 должно попасть в '0%'
# pd.cut с bins=[0, 0.0001, ...] и include_lowest=True поместит 0 в первый интервал
df.loc[df['discount'] == 0, 'discount_bin'] = '0%'

print("\n=== Распределение по discount_bin ===")
print(df['discount_bin'].value_counts().sort_index())

# === 6. Итоговая проверка схемы ===
print("\n=== Итоговая схема данных ===")
print(df.dtypes)
print(f"\nРазмерность после преобразований: {df.shape}")
print(f"Новые столбцы: {[c for c in df.columns if c not in ['order_id', 'order_date', 'customer_id', 'product_category', 'region', 'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days', 'customer_rating', 'revenue']]}")

# === 7. Сохранение очищенного датасета ===
df.to_csv('ecommerce_sales_clean.csv', index=False, encoding='utf-8')
print("\nОчищенный датасет сохранён: ecommerce_sales_clean.csv")

=== Описательная статистика числовых полей ===
        quantity  unit_price   discount  delivery_days  customer_rating  \
count  5000.0000   5000.0000  5000.0000      5000.0000        5000.0000   
mean      4.0448    308.4188     0.1800         6.1188           2.9740   
std       2.0204    169.2594     0.1014         3.1533           1.1577   
min       1.0000     15.1500     0.0000         1.0000           1.0000   
25%       2.0000    161.8950     0.0900         3.0000           2.0000   
50%       4.0000    309.8900     0.1800         6.0000           3.0000   
75%       6.0000    455.5575     0.2700         9.0000           4.0000   
max       7.0000    599.9600     0.3500        11.0000           5.0000   

         revenue  
count  5000.0000  
mean   1021.9551  
std     825.5842  
min      11.2100  
25%     354.5275  
50%     796.6500  
75%    1515.6900  
max    4119.3300  

=== Проверка бизнес-ограничений ===
quantity <= 0: 0 записей
unit_price <= 0: 0 записей
discount вне [0.0

### Промежуточный вывод по Этапу 3: Бизнес-валидация и очистка

На основе фактических результатов выполнения кода в notebook, бизнес-валидация и очистка датасета успешно завершены. Все данные соответствуют ожидаемым бизнес-правилам, аномалий, требующих удаления или импутации, не обнаружено.

#### 1. Числовые поля и бизнес-ограничения
Описательная статистика и проверки границ подтвердили абсолютную корректность числовых данных:
*   **`quantity`**: диапазон `[1, 7]`, среднее `4.04`. Нарушений (`<= 0`) — **0**.
3.  **`unit_price`**: диапазон `[15.15, 599.96]`, среднее `308.42`. Нарушений (`<= 0`) — **0**.
*   **`discount`**: диапазон `[0.0, 0.35]`, среднее `0.18`. Нарушений (вне `[0.0, 1.0]`) — **0**.
*   **`revenue`**: диапазон `[11.21, 4119.33]`, среднее `1021.96`. Нарушений (`< 0`) — **0**.
*   **`delivery_days`**: диапазон `[1, 11]`, среднее `6.12`. Нарушений (`< 0`) — **0**.
*   **`customer_rating`**: диапазон `[1.0, 5.0]`, среднее `2.97`. Нарушений (вне `[1.0, 5.0]`) — **0**.

**Вывод:** Математическая и логическая целостность числовых показателей составляет 100%.

#### 2. Категориальные поля и проверка регистра
Состав категорий полностью соответствует метаданным:
*   **`product_category`** (4 уникальных): Electronics (1777), Clothing (1531), Home (969), Beauty (723). Все значения в Title Case.
*   **`region`** (4 уникальных): West (1289), North (1276), East (1227), South (1208). Все значения в Title Case.
*   **`payment_method`** (3 уникальных): Card (2270), COD (1774), Wallet (956). 
    *   *Важное наблюдение:* Проверка регистра выявила **1774** значения, не соответствующих строгому Title Case. Это количество в точности совпадает с частотой значения **`COD`** (Cash on Delivery). Поскольку `COD` является устоявшейся бизнес-аббревиатурой, а не опечаткой (например, "cod" или "Cod"), **это не является ошибкой данных** и не требует исправления.

#### 3. Временной диапазон (`order_date`)
*   Преобразование в `datetime64[us]` прошло успешно, пропусков (NaT) — **0**.
*   **Минимальная дата**: `2022-01-01`
*   **Максимальная дата**: `2035-09-09`
*   **Диапазон**: 4999 дней.
*   **Даты из будущего** (относительно 2026-11-03): **3232** записи. Это ожидаемая характеристика данного синтетического/прогнозного датасета, аномалией не считается.

#### 4. Производные признаки
Успешно созданы и валидированы два новых поля:
*   **`discount_pct`**: непрерывная переменная (диапазон 0.0 – 35.0).
*   **`discount_bin`** (категориальная, 5 уровней):
    *   `0%`: 68 записей (1.36%)
    *   `1-10%`: 1333 записи (26.66%)
    *   `11-20%`: 1401 запись (28.02%)
    *   `21-30%`: 1502 записи (30.04%)
    *   `>30%`: 696 записей (13.92%)
    *   *Сумма:* 5000 записей. Распределение показывает, что большинство заказов (около 58%) имеют скидку в диапазоне 11-30%.

#### 5. Итоговая схема и артефакты
*   **Размерность**: `(5000, 14)`
*   **Типы данных**: `order_date` успешно преобразован в `datetime64[us]`, `discount_bin` имеет тип `category`. Остальные типы сохранены корректно.
*   **Артефакт**: Очищенный и обогащенный датасет успешно сохранён в файл `ecommerce_sales_clean.csv`.

---